# Avazu CTR Prediction

Predicting click-through rate on 40 million mobile ad impressions. All models built from scratch using NumPy only.

In [ ]:
%matplotlib inline
import csv, os, pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

DATA_PATH = 'train'
OUT_DIR   = '.'

In [ ]:
# ── All function and class definitions ──────────────────────────────────────

def reservoir_sample(filepath, k=200_000, seed=42):
    rng = np.random.default_rng(seed)
    reservoir = []
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if i < k:
                reservoir.append(row)
            else:
                j = int(rng.integers(0, i + 1))
                if j < k:
                    reservoir[j] = row
            if (i + 1) % 5_000_000 == 0:
                print(f'  ... streamed {(i+1)//1_000_000}M rows')
    print(f'  Sampled {len(reservoir):,} rows from {i+1:,} total')
    return reservoir


def run_eda(rows):
    n      = len(rows)
    clicks = sum(int(r['click']) for r in rows)
    print(f'Total rows   : {n:,}')
    print(f'Total clicks : {clicks:,}')
    print(f'Overall CTR  : {100*clicks/n:.2f}%')
    cat_cols = [
        'C1','banner_pos','site_id','site_domain','site_category',
        'app_id','app_domain','app_category','device_id','device_ip',
        'device_model','device_type','device_conn_type',
        'C14','C15','C16','C17','C18','C19','C20','C21'
    ]
    print(f'\n{"Column":<20} {"Unique":>8}  {"Top value":>15}  {"Rows":>8}  {"CTR":>6}')
    print('-' * 68)
    for col in cat_cols:
        counter = Counter()
        ctr_map = {}
        for r in rows:
            v = r[col]
            counter[v] += 1
            if v not in ctr_map:
                ctr_map[v] = [0, 0]
            ctr_map[v][0] += int(r['click'])
            ctr_map[v][1] += 1
        top_val, top_cnt = counter.most_common(1)[0]
        top_ctr = 100 * ctr_map[top_val][0] / ctr_map[top_val][1]
        print(f'{col:<20} {len(counter):>8}  {top_val:>15}  {top_cnt:>8}  {top_ctr:>5.1f}%')


def plot_eda(rows):
    hour_stats = {}
    for r in rows:
        h = int(r['hour']) % 100
        if h not in hour_stats:
            hour_stats[h] = [0, 0]
        hour_stats[h][0] += int(r['click'])
        hour_stats[h][1] += 1
    hours = sorted(hour_stats)
    ctrs  = [100 * hour_stats[h][0] / hour_stats[h][1] for h in hours]
    n      = len(rows)
    clicks = sum(int(r['click']) for r in rows)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(hours, ctrs, color='#378ADD', edgecolor='none')
    axes[0].axhline(y=16.98, color='#E24B4A', linestyle='--', linewidth=1, label='Overall CTR 16.98%')
    axes[0].set_xlabel('Hour of day')
    axes[0].set_ylabel('CTR (%)')
    axes[0].set_title('CTR by hour of day')
    axes[0].legend(fontsize=9)
    axes[0].set_xticks(range(0, 24, 2))
    axes[1].bar(['No click (83%)', 'Click (17%)'], [n - clicks, clicks],
                color=['#888780', '#1D9E75'], edgecolor='none')
    axes[1].set_ylabel('Row count')
    axes[1].set_title('Class imbalance')
    for i, v in enumerate([n - clicks, clicks]):
        axes[1].text(i, v + 500, f'{v:,}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()


def time_based_split(rows):
    train_rows, val_rows, test_rows = [], [], []
    for r in rows:
        day = (int(r['hour']) // 100) % 100
        if day <= 28:
            train_rows.append(r)
        elif day == 29:
            val_rows.append(r)
        else:
            test_rows.append(r)
    def ctr(s):
        return 100 * sum(r['click'] == '1' for r in s) / len(s) if s else 0.0
    print(f'  Train (Oct 21-28): {len(train_rows):,} rows  CTR={ctr(train_rows):.2f}%')
    print(f'  Val   (Oct 29)   : {len(val_rows):,} rows  CTR={ctr(val_rows):.2f}%')
    print(f'  Test  (Oct 30)   : {len(test_rows):,} rows  CTR={ctr(test_rows):.2f}%  <- holdout, opened once at the end')
    return train_rows, val_rows, test_rows


class FeatureEngineer:
    FREQ_COLS   = ['site_id','site_domain','site_category','app_id','app_domain',
                   'app_category','device_id','device_ip','device_model',
                   'C14','C17','C19','C20','C21']
    ONEHOT_COLS = ['C1','C18','device_type','device_conn_type','banner_pos','C15','C16']

    def __init__(self):
        self.freq_maps    = {}
        self.onehot_maps  = {}
        self.scale_params = {}
        self.feature_names = []
        self.fitted = False

    def fit(self, rows):
        for col in self.FREQ_COLS:
            self.freq_maps[col] = Counter(r[col] for r in rows)
        for col in self.ONEHOT_COLS:
            self.onehot_maps[col] = sorted(set(r[col] for r in rows))
        self.fitted = True

    def _row_to_features(self, row):
        feats = []
        h_raw       = int(row['hour'])
        hour_of_day = h_raw % 100
        day_of_week = (h_raw // 100) % 100 % 7
        feats.append(float(hour_of_day))
        feats.append(float(day_of_week))
        for col in self.FREQ_COLS:
            feats.append(np.log1p(self.freq_maps[col].get(row[col], 0)))
        for col in self.ONEHOT_COLS:
            v = row[col]
            for cat in self.onehot_maps[col][1:]:  # drop first category (reference)
                feats.append(1.0 if v == cat else 0.0)
        return feats

    def transform(self, rows):
        X = np.array([self._row_to_features(r) for r in rows], dtype=np.float32)
        if not self.feature_names:
            self.feature_names = ['hour_of_day', 'day_of_week']
            self.feature_names += [f'freq_{c}' for c in self.FREQ_COLS]
            for col in self.ONEHOT_COLS:
                self.feature_names += [f'{col}_{v}' for v in self.onehot_maps[col][1:]]
        return X

    def fit_transform(self, rows):
        self.fit(rows)
        X = self.transform(rows)
        n_scale = 2 + len(self.FREQ_COLS)
        means = X[:, :n_scale].mean(axis=0)
        stds  = X[:, :n_scale].std(axis=0) + 1e-8
        self.scale_params = {'mean': means, 'std': stds, 'n': n_scale}
        X[:, :n_scale] = (X[:, :n_scale] - means) / stds
        return X

    def transform_scaled(self, rows):
        X = self.transform(rows)
        n = self.scale_params['n']
        X[:, :n] = (X[:, :n] - self.scale_params['mean']) / self.scale_params['std']
        return X


def log_loss(y_true, y_prob, eps=1e-7):
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

def confusion_matrix(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return np.array([[tn, fp], [fn, tp]])

def precision_recall_f1(y_true, y_pred):
    cm   = confusion_matrix(y_true, y_pred)
    tp   = cm[1, 1]; fp = cm[0, 1]; fn = cm[1, 0]
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return prec, rec, f1

def roc_auc(y_true, scores, n_thresholds=300):
    thresholds = np.linspace(0, 1, n_thresholds)
    tprs, fprs = [], []
    pos = y_true.sum(); neg = len(y_true) - pos
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tp = ((pred == 1) & (y_true == 1)).sum()
        fp = ((pred == 1) & (y_true == 0)).sum()
        tprs.append(tp / (pos + 1e-8))
        fprs.append(fp / (neg + 1e-8))
    tprs, fprs = np.array(tprs), np.array(fprs)
    order = np.argsort(fprs)
    trapezoid = getattr(np, 'trapezoid', np.trapz)
    return trapezoid(tprs[order], fprs[order])

def pr_auc(y_true, scores, n_thresholds=300):
    thresholds = np.linspace(0, 1, n_thresholds)
    precisions, recalls = [], []
    for t in thresholds:
        pred = (scores >= t).astype(int)
        prec, rec, _ = precision_recall_f1(y_true, pred)
        precisions.append(prec); recalls.append(rec)
    precisions, recalls = np.array(precisions), np.array(recalls)
    order = np.argsort(recalls)
    trapezoid = getattr(np, 'trapezoid', np.trapz)
    return float(trapezoid(precisions[order], recalls[order]))

def find_optimal_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.60, 200)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        _, _, f1 = precision_recall_f1(y_true, pred)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t

def print_eval(name, y_true, y_prob, threshold=0.5):
    y_pred        = (y_prob >= threshold).astype(int)
    ll            = log_loss(y_true, y_prob)
    auc           = roc_auc(y_true, y_prob)
    prauc         = pr_auc(y_true, y_prob)
    prec, rec, f1 = precision_recall_f1(y_true, y_pred)
    cm            = confusion_matrix(y_true, y_pred)
    print(f'  {name}  (threshold={threshold:.3f})')
    print(f'  Log-loss : {ll:.4f}   AUC : {auc:.4f}   PR-AUC : {prauc:.4f}')
    print(f'  Precision: {prec:.4f}   Recall: {rec:.4f}   F1: {f1:.4f}')
    print(f'  Confusion matrix:  TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}')
    return {'log_loss': ll, 'auc': auc, 'pr_auc': prauc,
            'precision': prec, 'recall': rec, 'f1': f1, 'threshold': threshold}


class NaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    def fit(self, X, y):
        n, d   = X.shape
        n_pos  = int(y.sum()); n_neg = n - n_pos
        self.log_prior = np.array([np.log(n_neg / n), np.log(n_pos / n)])
        X_bin  = (X > 0).astype(np.float32)
        counts = np.zeros((d, 2))
        counts[:, 0] = X_bin[y == 0].sum(axis=0)
        counts[:, 1] = X_bin[y == 1].sum(axis=0)
        totals = np.array([n_neg, n_pos])
        self.log_like     = np.log((counts + self.alpha) / (totals + 2 * self.alpha))
        self.log_like_neg = np.log(1 - np.exp(self.log_like))
        print(f'  Fitted on {n:,} samples, {d} features')
    def predict_proba(self, X):
        X_bin = (X > 0).astype(np.float32)
        ll1   = X_bin @ self.log_like[:, 1] + (1 - X_bin) @ self.log_like_neg[:, 1]
        ll0   = X_bin @ self.log_like[:, 0] + (1 - X_bin) @ self.log_like_neg[:, 0]
        log_sum = np.logaddexp(self.log_prior[0] + ll0, self.log_prior[1] + ll1)
        return np.exp(self.log_prior[1] + ll1 - log_sum)


class LogisticRegression:
    def __init__(self, lr=0.1, lambda_=0.001, reg='l2', batch_size=2048, n_epochs=20, decay=0.5):
        self.lr=lr; self.lambda_=lambda_; self.reg=reg
        self.batch_size=batch_size; self.n_epochs=n_epochs; self.decay=decay
    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))
    def fit(self, X, y, X_val=None, y_val=None, verbose=True):
        n, d   = X.shape
        self.w = np.zeros(d, dtype=np.float32)
        self.b = np.float32(0.0)
        rng    = np.random.default_rng(42)
        history = {'train_loss': [], 'val_loss': []}
        for epoch in range(self.n_epochs):
            idx    = rng.permutation(n)
            X_shuf = X[idx]; y_shuf = y[idx]
            lr_t   = self.lr / (1.0 + self.decay * epoch)
            epoch_loss, n_batches = 0.0, 0
            for start in range(0, n, self.batch_size):
                Xb  = X_shuf[start:start + self.batch_size]
                yb  = y_shuf[start:start + self.batch_size]
                p   = self._sigmoid(Xb @ self.w + self.b)
                err = p - yb
                penalty = self.lambda_ * (np.sign(self.w) if self.reg == 'l1' else self.w)
                self.w -= lr_t * ((Xb.T @ err) / len(yb) + penalty)
                self.b -= lr_t * float(err.mean())
                epoch_loss += log_loss(yb, p); n_batches += 1
            tl = epoch_loss / n_batches
            history['train_loss'].append(tl)
            if X_val is not None:
                vl = log_loss(y_val, self.predict_proba(X_val))
                history['val_loss'].append(vl)
                if verbose and (epoch + 1) % 5 == 0:
                    print(f'    epoch {epoch+1:3d}  train={tl:.4f}  val={vl:.4f}  lr={lr_t:.5f}')
        self.history = history
        return self
    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)
    def top_features(self, feature_names, n=10):
        idx = np.argsort(np.abs(self.w))[::-1][:n]
        return [(feature_names[i], float(self.w[i])) for i in idx]


class DecisionTree:
    class _Node:
        __slots__ = ['feature','threshold','left','right','value']
        def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
            self.feature=feature; self.threshold=threshold
            self.left=left; self.right=right; self.value=value
    def __init__(self, max_depth=6, min_samples_leaf=50, n_thresholds=20):
        self.max_depth=max_depth; self.min_samples_leaf=min_samples_leaf
        self.n_thresholds=n_thresholds; self.root=None
    @staticmethod
    def _gini(y):
        if len(y) == 0: return 0.0
        p = y.mean(); return 1.0 - p*p - (1.0-p)*(1.0-p)
    def _best_split(self, X, y):
        n=len(y); parent_gini=self._gini(y)
        best_gain, best_feat, best_thresh = -1.0, None, None
        for feat in range(X.shape[1]):
            vals = X[:, feat]
            thresholds = np.unique(np.percentile(vals, np.linspace(5, 95, self.n_thresholds)))
            for thresh in thresholds:
                left = vals <= thresh
                nl = int(left.sum()); nr = n - nl
                if nl < self.min_samples_leaf or nr < self.min_samples_leaf: continue
                gl = self._gini(y[left]); gr = self._gini(y[~left])
                gain = parent_gini - (nl*gl + nr*gr) / n
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, feat, thresh
        return best_feat, best_thresh
    def _build(self, X, y, depth):
        if depth >= self.max_depth or len(y) < 2*self.min_samples_leaf:
            return self._Node(value=float(y.mean()))
        feat, thresh = self._best_split(X, y)
        if feat is None: return self._Node(value=float(y.mean()))
        mask  = X[:, feat] <= thresh
        left  = self._build(X[mask],  y[mask],  depth+1)
        right = self._build(X[~mask], y[~mask], depth+1)
        return self._Node(feature=feat, threshold=thresh, left=left, right=right)
    def fit(self, X, y):
        print(f'  Building tree (max_depth={self.max_depth}, min_samples_leaf={self.min_samples_leaf})...')
        self.root = self._build(X, y.astype(np.float32), 0); return self
    def _predict_row(self, x, node):
        if node.value is not None: return node.value
        if x[node.feature] <= node.threshold: return self._predict_row(x, node.left)
        return self._predict_row(x, node.right)
    def predict_proba(self, X):
        return np.array([self._predict_row(x, self.root) for x in X])


class RandomForest:
    def __init__(self, n_trees=10, max_depth=6, min_samples_leaf=50, n_thresholds=20, max_features='sqrt', seed=42):
        self.n_trees=n_trees; self.max_depth=max_depth
        self.min_samples_leaf=min_samples_leaf; self.n_thresholds=n_thresholds
        self.max_features=max_features; self.seed=seed
        self.trees=[]; self.feature_subsets=[]
    def fit(self, X, y):
        n, d = X.shape
        rng  = np.random.default_rng(self.seed)
        n_feat = int(np.sqrt(d)) if self.max_features == 'sqrt' else d
        self.trees=[]; self.feature_subsets=[]
        for i in range(self.n_trees):
            boot_idx = rng.integers(0, n, size=n)
            X_boot   = X[boot_idx]; y_boot = y[boot_idx]
            feat_idx = rng.choice(d, size=n_feat, replace=False)
            X_sub    = X_boot[:, feat_idx]
            tree = DecisionTree(max_depth=self.max_depth,
                                min_samples_leaf=self.min_samples_leaf,
                                n_thresholds=self.n_thresholds)
            print(f'  Tree {i+1}/{self.n_trees}  (bootstrap n={n:,}, features={n_feat}/{d})')
            tree.root = tree._build(X_sub, y_boot.astype(np.float32), 0)
            self.trees.append(tree); self.feature_subsets.append(feat_idx)
        return self
    def predict_proba(self, X):
        preds = np.zeros(len(X), dtype=np.float64)
        for tree, feat_idx in zip(self.trees, self.feature_subsets):
            preds += tree.predict_proba(X[:, feat_idx])
        return preds / self.n_trees


def grid_search(X_train, y_train, X_val, y_val):
    lrs=[0.5, 0.1, 0.01]; lambdas=[0.0001, 0.001, 0.01]; regs=['l2','l1']
    print(f'  {"reg":>4}  {"lr":>6}  {"lambda":>8}  {"val_loss":>10}  {"val_auc":>9}')
    print('  ' + '-'*46)
    best_loss, best_params = float('inf'), {}
    for reg in regs:
        for lr in lrs:
            for lam in lambdas:
                m     = LogisticRegression(lr=lr, lambda_=lam, reg=reg, n_epochs=20)
                m.fit(X_train, y_train, verbose=False)
                probs = m.predict_proba(X_val)
                vl    = log_loss(y_val, probs); va = roc_auc(y_val, probs)
                mark  = ' <-- best' if vl < best_loss else ''
                print(f'  {reg:>4}  {lr:>6.3f}  {lam:>8.4f}  {vl:>10.4f}  {va:>9.4f}{mark}')
                if vl < best_loss:
                    best_loss=vl; best_params={'lr':lr,'lambda_':lam,'reg':reg}
    print(f'  Best: reg={best_params["reg"]}  lr={best_params["lr"]}  lambda={best_params["lambda_"]}  val_loss={best_loss:.4f}')
    return best_params


def slice_analysis(rows, y_true, y_prob, label=''):
    print(f'  Slice analysis  {label}')
    print(f'  {"Slice":<28}  {"N":>6}  {"CTR":>6}  {"AUC":>7}  {"LogLoss":>9}')
    print('  ' + '-'*62)
    for dt in sorted(set(r['device_type'] for r in rows)):
        mask = np.array([r['device_type'] == dt for r in rows])
        if mask.sum() < 50: continue
        yt, yp = y_true[mask], y_prob[mask]
        auc = roc_auc(yt, yp) if yt.sum() > 0 else float('nan')
        ll  = log_loss(yt, yp)
        print(f'  {"device_type="+dt:<28}  {mask.sum():>6}  {yt.mean()*100:>5.1f}%  {auc:>7.4f}  {ll:>9.4f}')
    hour_groups = {
        'night  (00-05)': lambda h: h < 6,
        'morning(06-11)': lambda h: 6 <= h < 12,
        'afternoon(12-17)': lambda h: 12 <= h < 18,
        'evening(18-23)': lambda h: h >= 18,
    }
    for name, fn in hour_groups.items():
        mask = np.array([fn(int(r['hour']) % 100) for r in rows])
        if mask.sum() < 50: continue
        yt, yp = y_true[mask], y_prob[mask]
        auc = roc_auc(yt, yp) if yt.sum() > 0 else float('nan')
        ll  = log_loss(yt, yp)
        print(f'  {name:<28}  {mask.sum():>6}  {yt.mean()*100:>5.1f}%  {auc:>7.4f}  {ll:>9.4f}')

print('All functions loaded.')

---
## Step 1 — Data sampling

**Problem:** The raw dataset has 40 million rows. Loading everything into memory requires ~8 GB — more than most laptops have available.

**Why not read the first 200k rows?** The data is sorted chronologically. The first 200k rows are all from Oct 21 at midnight — the model would never see later dates or hours.

**Solution:** Reservoir sampling — stream all 40M rows one at a time, keep a 200k random sample where every row had equal probability of selection.

In [ ]:
ROWS_CACHE = 'rows_cache.pkl'

if os.path.exists(ROWS_CACHE):
    with open(ROWS_CACHE, 'rb') as f:
        rows = pickle.load(f)
    print(f'Loaded {len(rows):,} rows from cache (skipping 40M row stream)')
else:
    print('Streaming 40M rows...')
    rows = reservoir_sample(DATA_PATH, k=200_000)
    with open(ROWS_CACHE, 'wb') as f:
        pickle.dump(rows, f)
    print('Cached for future runs')

**Finding:** 200,000 rows sampled from 40,428,967 total. ~20k rows per day across all 10 days (Oct 21–30) — confirmed representative by the date distribution check in EDA.

---
## Step 2 — EDA (Exploratory Data Analysis)

**Problem:** Before building any model, we need to understand what the data looks like — class balance, column cardinalities, any anomalies.

**Solution:** Print stats on every column. Check CTR by hour of day. Verify reservoir sample is spread across all dates.

In [ ]:
run_eda(rows)

In [ ]:
plot_eda(rows)

**Findings:**
- CTR = 16.98% — class imbalance. 83% of rows are non-clicks. Accuracy is a misleading metric here.
- `device_id = a99f214a` appears in 165k of 200k rows — anonymous user placeholder, not a real device ID.
- `C20` contains -1 values — Avazu's null marker.
- Hour CTR ranges 15.6%–18.7% — modest time-of-day signal.
- App traffic (~20% CTR) vs site traffic (~13% CTR) — content type matters.

---
## Step 3 — Time-based split

**Problem:** A random 80/20 split would let the model train on October 29 rows while evaluating on October 22 rows. In production, you always predict the future from the past. Training on future data is **temporal leakage** — the model looks better than it really is.

**Solution:** Split strictly by date.

In [ ]:
train_rows, val_rows, test_rows = time_based_split(rows)

**Finding:** Train Oct 21–28 (~160k rows), Val Oct 29 (~19k), Test Oct 30 (~21k).

The test set is never touched until the final step. Any decision made by looking at test results — including threshold selection — is leakage.

---
## Step 4 — Feature engineering

**Problem:** Raw data contains text IDs like `1fbe01fe` and category codes. Models can only do math on numbers.

**Solution:** Three encoding techniques — all fit on training data only, applied to val/test without refitting.

In [ ]:
fe      = FeatureEngineer()
X_train = fe.fit_transform(train_rows)   # learns encoding maps from train only
X_val   = fe.transform_scaled(val_rows)  # applies the same maps
X_test  = fe.transform_scaled(test_rows) # same

y_train = np.array([int(r['click']) for r in train_rows], dtype=np.float32)
y_val   = np.array([int(r['click']) for r in val_rows],   dtype=np.float32)
y_test  = np.array([int(r['click']) for r in test_rows],  dtype=np.float32)

print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')
print(f'{len(fe.feature_names)} features total')
print(f'  2 time features (hour_of_day, day_of_week)')
print(f'  14 frequency-encoded columns')
print(f'  43 one-hot encoded columns (from 7 low-cardinality columns)')

**Finding:** 24 raw columns → 59 numeric features.

- **Frequency encoding** (14 cols): replace each category ID with `log(count + 1)` from training. site_id has ~1,800 unique values — frequency encoding keeps it as 1 column instead of 1,800.
- **One-hot encoding** (7 cols): device_type has 4 values. Keeping as integer implies false ordering. One-hot creates 4 binary columns.
- **Z-score scaling**: mean 0, std 1. Fit on train only — no leakage.

---
## Step 5 — Naive Bayes (baseline)

**Problem:** Before building complex models, establish a floor. The baseline tells you whether improvements are real.

**Solution:** Bernoulli Naive Bayes with Laplace smoothing. Intentionally weak — binarizes all features at 0, discarding the frequency encoding values.

In [ ]:
nb = NaiveBayes(alpha=1.0)
nb.fit(X_train, y_train)
nb_probs  = nb.predict_proba(X_val)
nb_thresh = find_optimal_threshold(y_val, nb_probs)

print('--- Default threshold (0.5) ---')
nb_results = print_eval('Naive Bayes', y_val, nb_probs, threshold=0.5)

print('\n--- Optimal threshold (tuned on val) ---')
nb_results_t = print_eval('Naive Bayes', y_val, nb_probs, threshold=nb_thresh)

**Finding:** Log-loss 0.539, AUC 0.625 — this is the floor. Every model after this must beat it.

Binarizing features is the key weakness: a popular site (freq=8.2) and a rare site (freq=0.7) both become 1. All the frequency encoding information is thrown away.

---
## Step 6 — Logistic Regression

**Problem:** Naive Bayes binarizes features and assumes independence. Logistic Regression uses actual feature values and learns a weighted combination.

**Solution:** Mini-batch SGD with L1/L2 regularization and learning rate decay. Grid search over 18 hyperparameter combinations.

In [ ]:
print('Grid search (18 combinations)...')
best_params = grid_search(X_train, y_train, X_val, y_val)

print('\nRetraining best config...')
best_lr = LogisticRegression(**best_params, n_epochs=20)
best_lr.fit(X_train, y_train, X_val=X_val, y_val=y_val, verbose=True)

lr_val_probs = best_lr.predict_proba(X_val)
lr_thresh    = find_optimal_threshold(y_val, lr_val_probs)

print('\n--- Default threshold (0.5) ---')
lr_results = print_eval(f'Logistic Regression [{best_params["reg"].upper()}]', y_val, lr_val_probs, threshold=0.5)

print('\n--- Optimal threshold ---')
lr_results_t = print_eval(f'Logistic Regression [{best_params["reg"].upper()}]', y_val, lr_val_probs, threshold=lr_thresh)

print('\nTop 10 features by weight:')
for feat, w in best_lr.top_features(fe.feature_names, n=10):
    direction = '-> click' if w > 0 else '-> no click'
    print(f'  {feat:<30}  w={w:+.4f}  {direction}')

**Finding:** Log-loss 0.411, AUC 0.661 — 23.7% improvement over Naive Bayes.

L1 and L2 regularization tied, meaning most features are genuinely contributing signal. Top features: C18=1 strongly predicts no-click (w=−0.73), popular sites predict click (freq_site_id w=+0.26).

---
## Step 7 — Decision Tree

**Problem:** Logistic Regression only captures linear relationships — it draws one straight boundary through feature space. Real click patterns may involve combinations of features.

**Solution:** Decision Tree with Gini impurity. Asks yes/no questions to recursively split data. Regularized with max_depth=6 and min_samples_leaf=50 to prevent overfitting.

In [ ]:
dt = DecisionTree(max_depth=6, min_samples_leaf=50)
dt.fit(X_train, y_train)

dt_val_probs = dt.predict_proba(X_val)
dt_thresh    = find_optimal_threshold(y_val, dt_val_probs)

print('--- Default threshold (0.5) ---')
dt_results = print_eval('Decision Tree', y_val, dt_val_probs, threshold=0.5)

print('\n--- Optimal threshold ---')
dt_results_t = print_eval('Decision Tree', y_val, dt_val_probs, threshold=dt_thresh)

**Finding:** Log-loss 0.407, AUC 0.671 — best single model on both metrics.

The non-linear splits capture interactions between features that logistic regression cannot (e.g. popular site AND mobile device AND evening hour).

---
## Step 8 — Random Forest

**Problem:** A single Decision Tree has high variance — small changes to training data produce a very different tree.

**Solution:** Train 10 trees, each on a random bootstrap sample with random feature subsets. Average their predictions. Decorrelated trees → reduced variance.

In [ ]:
rf = RandomForest(n_trees=10, max_depth=6, min_samples_leaf=50)
rf.fit(X_train, y_train)

rf_val_probs = rf.predict_proba(X_val)
rf_thresh    = find_optimal_threshold(y_val, rf_val_probs)

print('--- Default threshold (0.5) ---')
rf_results = print_eval('Random Forest', y_val, rf_val_probs, threshold=0.5)

print('\n--- Optimal threshold ---')
rf_results_t = print_eval('Random Forest', y_val, rf_val_probs, threshold=rf_thresh)

**Finding:** AUC 0.677, PR-AUC 0.288 — highest of all models on both metrics.

Two sources of randomness: bootstrap sampling (~63% unique rows per tree) and feature subsampling (√59 ≈ 7 features per split). This forces trees to be different from each other.

---
## Step 9 — Threshold tuning

**Problem:** The default threshold of 0.5 gives near-zero recall. With 17% CTR, model outputs cluster around 0.15–0.25 — nothing crosses 0.5, so the model almost never predicts click=1.

**Solution:** Sweep thresholds 0.05–0.60 on the val set. Pick the threshold that maximises F1.

In [ ]:
results   = {'Naive Bayes': nb_results,   'Logistic Regression': lr_results,
             'Decision Tree': dt_results,  'Random Forest': rf_results}
results_t = {'Naive Bayes': nb_results_t, 'Logistic Regression': lr_results_t,
             'Decision Tree': dt_results_t,'Random Forest': rf_results_t}

print(f'  {"Model":<28}  {"Default F1":>10}  {"Optimal t":>10}  {"Optimal F1":>11}')
print('  ' + '-'*64)
for name in results:
    print(f'  {name:<28}  {results[name]["f1"]:>10.4f}  '
          f'{results_t[name]["threshold"]:>10.3f}  {results_t[name]["f1"]:>11.4f}')

**Finding:** At threshold 0.5, LR recall = 0.3%. At optimal threshold (~0.196), recall jumps to 52.8%.

The optimal threshold is close to the actual CTR of 17% — a calibrated model's average prediction should be near the true positive rate.

---
## Visualizations — ROC curves and model comparison

In [ ]:
probs_dict = {'Naive Bayes': nb_probs, 'Logistic Regression': lr_val_probs,
              'Decision Tree': dt_val_probs, 'Random Forest': rf_val_probs}
colors = ['#888780', '#378ADD', '#1D9E75', '#BA7517']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax_idx, (metric, xlabel, ylabel, title) in enumerate([
    ('roc', 'False positive rate', 'True positive rate', 'ROC curves'),
    ('pr',  'Recall',              'Precision',          'Precision-Recall curves')
]):
    for (name, y_prob), color in zip(probs_dict.items(), colors):
        thresholds = np.linspace(0, 1, 300)
        xs, ys = [], []
        pos = y_val.sum(); neg = len(y_val) - pos
        for t in thresholds:
            pred = (y_prob >= t).astype(int)
            tp = ((pred==1)&(y_val==1)).sum(); fp = ((pred==1)&(y_val==0)).sum()
            fn = ((pred==0)&(y_val==1)).sum()
            if metric == 'roc':
                xs.append(fp/(neg+1e-8)); ys.append(tp/(pos+1e-8))
            else:
                prec = tp/(tp+fp+1e-8); rec = tp/(tp+fn+1e-8)
                xs.append(rec); ys.append(prec)
        order = np.argsort(xs)
        axes[ax_idx].plot(np.array(xs)[order], np.array(ys)[order],
                          label=name, color=color, linewidth=1.5)
    if metric == 'roc':
        axes[ax_idx].plot([0,1],[0,1],'k--',linewidth=0.8,label='Random')
    axes[ax_idx].set_xlabel(xlabel); axes[ax_idx].set_ylabel(ylabel)
    axes[ax_idx].set_title(title); axes[ax_idx].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
names      = list(results.keys())
log_losses = [results[n]['log_loss'] for n in names]
aucs       = [results[n]['auc']      for n in names]

x = np.arange(len(names)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, log_losses, w, label='Log-loss (lower is better)', color='#378ADD', edgecolor='none')
ax.bar(x + w/2, aucs,       w, label='AUC (higher is better)',     color='#1D9E75', edgecolor='none')
for i, (ll, auc) in enumerate(zip(log_losses, aucs)):
    ax.text(i - w/2, ll + 0.005, f'{ll:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, auc + 0.005, f'{auc:.3f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(0.35, 0.75); ax.set_title('Model comparison — validation set')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Step 10 — Slice analysis

**Problem:** Overall metrics hide subpopulation failures. A model with AUC 0.677 overall might be excellent on phones but terrible on tablets.

**Solution:** Break performance down by device_type and hour group.

In [ ]:
slice_analysis(val_rows, y_val, dt_val_probs, label='Decision Tree')

**Finding:** device_type=0 (tablet) has AUC 0.57 — the model struggles here, likely due to fewer training examples. device_type=1 (phone) dominates the dataset and performs well. Evening hours (18–23) show slightly higher AUC across all models.

---
## Step 11 — Final evaluation on holdout test set

**Problem:** Need an honest estimate of real-world performance on data the model has never seen.

**Solution:** The Oct 30 test set has been locked since Step 3. We select the best model by val log-loss, then evaluate exactly once.

In [ ]:
best_name = min(results, key=lambda k: results[k]['log_loss'])
print(f'Best model on val (by log-loss): {best_name}')

best_test_probs = {
    'Naive Bayes':        nb.predict_proba(X_test),
    'Logistic Regression': best_lr.predict_proba(X_test),
    'Decision Tree':       dt.predict_proba(X_test),
    'Random Forest':       rf.predict_proba(X_test),
}
best_thresh_map = {
    'Naive Bayes': nb_thresh, 'Logistic Regression': lr_thresh,
    'Decision Tree': dt_thresh, 'Random Forest': rf_thresh,
}

test_probs = best_test_probs[best_name]
best_t     = best_thresh_map[best_name]

print(f'\nFinal result on Oct 30 test set (threshold={best_t:.3f}):')
final = print_eval(f'{best_name} — TEST', y_test, test_probs, threshold=best_t)

nb_ll = results['Naive Bayes']['log_loss']
print(f'\nLog-loss reduction vs Naive Bayes baseline: {100*(nb_ll - final["log_loss"])/nb_ll:.1f}%')

**Finding:** Decision Tree on Oct 30 — Log-loss 0.421, AUC 0.677, F1 0.369.

**24.5% log-loss reduction** over the Naive Bayes baseline. The small val-to-test gap (0.407 → 0.421) confirms the model generalises to new data rather than overfitting to the validation period.